In [2]:
import pandas as pd 

In [3]:
movies = pd.read_csv('movies.csv')

genome_scores = pd.read_csv('genome-scores.csv')

genome_tags = pd.read_csv('genome-tags.csv')

print(movies.shape)
print(genome_scores.shape)
print(genome_tags.shape)

(27278, 3)
(11709768, 3)
(1128, 2)


In [5]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
genome_scores.head()

,movieId,tagId,relevance
0,1,1,0.02500
1,1,2,0.02500
2,1,3,0.05775
3,1,4,0.09675
4,1,5,0.14675


In [7]:
genome_tags.head()

,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [4]:
genome = genome_scores.merge(
    genome_tags , on ='tagId'
)

print(genome.head())

   movieId  tagId  relevance           tag
0        1      1    0.02500           007
1        1      2    0.02500  007 (series)
2        1      3    0.05775  18th century
3        1      4    0.09675         1920s
4        1      5    0.14675         1930s


In [12]:
movie_features = genome.pivot(
    index ='movieId',
    columns = 'tag',
    values='relevance'
)
print(movie_features.shape)



(10381, 1128)


In [15]:
movie_features.fillna(0,inplace =True)

In [22]:
from sklearn.preprocessing import StandardScaler

scaler= StandardScaler()
X = scaler.fit_transform(movie_features)


print(X.shape)

(10381, 1128)


In [18]:
import torch
X_tensor= torch.tensor(X, dtype=torch.float32)
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

dataset = TensorDataset(
    X_tensor,X_tensor
)

loader = DataLoader(dataset ,batch_size=128 ,shuffle =True)




In [24]:
import torch.nn  as nn
class autoencoder(nn.Module):
    def __init__(self,input_dim):
        super().__init__()

        self.encoder =nn.Sequential(
            nn.Linear(input_dim ,512),
            nn.ReLU(),

            nn.Linear(512,216),
            nn.ReLU(),
            nn.Linear(216,64)
        )

        self.decoder = nn.Sequential(
            nn.Linear(64,256),
            nn.ReLU(),

            nn.Linear(256,512),
            nn.ReLU(),

            nn.Linear(512,input_dim)


        )
    
    def forward(self,x):
        latent = self.encoder(x)

        reconstructed = self.decoder(latent)

        return reconstructed
    


    

print(model)

autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=1128, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=216, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=64, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1128, bias=True)
  )
)


In [25]:
input_dim = X.shape[1]
model = autoencoder(input_dim)

criterion  =nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)



In [27]:
epochs = 50 
for epoch in range(epochs):
    total_loss =0
    for batch_x ,_ in loader:
        optimizer.zero_grad()

        output = model(batch_x)
        loss = criterion( output , batch_x)

        loss.backward()

        optimizer.step()
        total_loss +=loss.item()


    print(
        f"Epoch {epoch+1}/{epochs}---->"
        f"{total_loss:.4f}"
    )


Epoch 1/50---->20.7238
Epoch 2/50---->20.6654
Epoch 3/50---->20.5095
Epoch 4/50---->20.4812
Epoch 5/50---->20.3754
Epoch 6/50---->20.2964
Epoch 7/50---->20.3619
Epoch 8/50---->20.3185
Epoch 9/50---->20.3109
Epoch 10/50---->20.2678
Epoch 11/50---->20.2308
Epoch 12/50---->20.1263
Epoch 13/50---->20.1183
Epoch 14/50---->20.0787
Epoch 15/50---->20.0568
Epoch 16/50---->20.1517
Epoch 17/50---->20.1348
Epoch 18/50---->20.1629
Epoch 19/50---->20.0007
Epoch 20/50---->19.8338
Epoch 21/50---->19.8592
Epoch 22/50---->19.8060
Epoch 23/50---->19.7601
Epoch 24/50---->19.8137
Epoch 25/50---->19.7695
Epoch 26/50---->19.6881
Epoch 27/50---->19.6409
Epoch 28/50---->19.6540
Epoch 29/50---->19.5911
Epoch 30/50---->19.7379
Epoch 31/50---->19.6259
Epoch 32/50---->19.5318
Epoch 33/50---->19.4987
Epoch 34/50---->19.5078
Epoch 35/50---->19.5558
Epoch 36/50---->19.5205
Epoch 37/50---->19.4504
Epoch 38/50---->19.3412
Epoch 39/50---->19.3856
Epoch 40/50---->19.4830
Epoch 41/50---->19.3941
Epoch 42/50---->19.2913
E

In [28]:
with torch.no_grad():
    reconstructed = model(X_tensor)

    mse = ((reconstructed-X_tensor)**2).mean()

print(mse.item())

0.2300528734922409


In [29]:
with torch.no_grad():
    embeddings = model.encoder(X_tensor)

embeddings.shape

torch.Size([10381, 64])

In [30]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=11,
    metric ='cosine'

)
knn.fit(embeddings.numpy())

,n_neighbors,11
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [31]:
movies[movies["title"].str.contains("Toy Story")]

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3027,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
15401,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
21981,106022,Toy Story of Terror (2013),Animation|Children|Comedy
24458,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy
24460,115879,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy
25461,120468,Toy Story Toons: Partysaurus Rex (2012),Animation|Children|Comedy
25463,120474,Toy Story That Time Forgot (2014),Animation|Children


In [33]:
movie_id = 1

idx = movie_features.index.get_loc(movie_id)

distances, indices = knn.kneighbors(
    embeddings[idx].reshape(1,-1)
)

for i in indices[0][1:]:
    rec_movie_id = movie_features.index[i]

    print(
        movies.loc[
            movies.movieId==rec_movie_id,
            'title'
        ].values[0]
        
    )


Toy Story 2 (1999)
Monsters, Inc. (2001)
Toy Story 3 (2010)
Bug's Life, A (1998)
Incredibles, The (2004)
Cars (2006)
Ice Age (2002)
Monsters University (2013)
Finding Nemo (2003)
Up (2009)


In [34]:
distances, indices = knn.kneighbors(
    embeddings[idx].reshape(1,-1),
    n_neighbors=11
)

for dist, i in zip(distances[0][1:], indices[0][1:]):
    movie_id = movie_features.index[i]

    title = movies.loc[
        movies.movieId == movie_id,
        "title"
    ].values[0]

    print(f"{title} --> {1-dist:.3f}")

Toy Story 2 (1999) --> 0.970
Monsters, Inc. (2001) --> 0.961
Toy Story 3 (2010) --> 0.911
Bug's Life, A (1998) --> 0.892
Incredibles, The (2004) --> 0.835
Cars (2006) --> 0.824
Ice Age (2002) --> 0.819
Monsters University (2013) --> 0.817
Finding Nemo (2003) --> 0.806
Up (2009) --> 0.806


In [35]:
import torch

torch.save(
    model.encoder.state_dict(),
    "movie_encoder.pt"
)

import pickle
with open("knn.pkl","wb") as f:
    pickle.dump(knn,f)


with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)


import numpy as np

np.save(
    "movie_embeddings.npy",
    embeddings
)
movie_features.index.to_series().to_csv(
    "movie_index.csv",
    index=False
)
movies.to_csv(
    "movies_metadata.csv",
    index=False
)